# TriPlane SIM V1 3D — Phase B: generate the surrogate dataset

Phase B for the **tri-plane (xy/yz/zx) PL surrogate**. Everything downstream is built
on a **voxelized scene**: `material_grid.npy` is the voxel model produced by Phase A
(`voxelize.py` for an indoor floor plan, `voxelize_city.py` for OSM + terrain). This
notebook runs the engine over that voxel grid for many Tx positions and stores the
**normalized PL volume** per (Tx, band).

**Why one dataset feeds three networks:** a plane is just a slice of a PL volume, so we
store volumes once and the trainer (`phase_c3_planes_train.ipynb`) slices them into the
three axis-aligned stacks on the fly — no plane-specific storage. **PL only** (no
arrival-time: the 3-D eikonal is not sliceable and the planar model never uses it).

**Scene-agnostic on purpose.** Point `ROOT` at any voxelized SIM V1 3D-style folder;
this is the path to a tri-plane surrogate that generalizes across scenes, so a NEW
imported/voxelized model can get its own dataset without a bespoke pipeline.

In [ ]:
#@title Setup: Drive, deps, load the VOXELIZED scene
import os, sys, json, time, glob
from pathlib import Path
import numpy as np

# ROOT holds the voxelized scene (material_grid.npy etc.) + engine_3d.py / dataset_3d.py
ROOT = "/content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D"  #@param {type:"string"}
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception:
    pass
ROOT = str(Path(ROOT).expanduser().resolve())
if not Path(ROOT, "engine_3d.py").is_file():
    raise FileNotFoundError(f"ROOT is not a SIM V1 3D folder: {ROOT}")
try:
    import skfmm  # engine_3d imports it at module load even though we skip the eikonal
except ImportError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-fmm"], check=True)
_phys2 = Path(ROOT).parent.parent / "2D" / "SIM"
if not (_phys2 / "physics_v2.py").is_file():
    _phys2 = Path(ROOT)
sys.path.insert(0, str(_phys2)); sys.path.insert(0, ROOT)
import dataset_3d as D
from engine_3d import load_scene

scene, man = load_scene(ROOT)
M      = np.load(f"{ROOT}/material_grid.npy")
inside = np.load(f"{ROOT}/inside_mask.npy")
valid  = np.load(f"{ROOT}/valid_tx_mask.npy")
norm   = D.load_norm(man)
OUT    = f"{ROOT}/dataset"; os.makedirs(OUT, exist_ok=True)
print("voxel grid ", M.shape, f"({int(np.prod(M.shape)):,} voxels, cell {man['cell_size_m']} m)")
print("interior   ", f"{int(inside.sum()):,} | valid Tx {int(valid.sum()):,}")
print("scene_sha  ", D.scene_sha(M), "| bands", man["freqs_mhz"])

In [ ]:
#@title Config
RUN_MODE     = "full"   #@param ["full", "smoke"]
N_POSITIONS  = 750      #@param {type:"integer"}
SHARD_POS    = 25       #@param {type:"integer"}
MIN_SPACING  = 3.0      #@param {type:"number"}   min Tx spacing (voxels)
TRAIN_BANDS  = [619.0, 1935.0, 2442.0, 3500.0, 5500.0, 6125.0]
SEED         = 0
if RUN_MODE == "smoke":
    N_POSITIONS, SHARD_POS = 16, 8
mb = int(np.prod(M.shape)) * 2 / 1e6
print(f"{N_POSITIONS} positions x {len(TRAIN_BANDS)} bands = {N_POSITIONS*len(TRAIN_BANDS)} samples  [{RUN_MODE}]")
print(f"projected PL store ~{N_POSITIONS*len(TRAIN_BANDS)*mb/1000:.1f} GB (fp16, no tau)")

## The voxel scene — the foundation everything slices

`material_grid.npy` is the voxelization of the model: each voxel is a material class.
The tri-plane surrogate never sees the raw geometry — only these voxel classes (as the
one-hot channels) and where the Tx sits. So the composition below is literally what the
networks learn to propagate through.

In [ ]:
#@title Voxel composition
mats = man["materials"]
counts = np.bincount(M.ravel(), minlength=len(mats))
print(f"{'id':>2}  {'material':<20}{'voxels':>11}{'% grid':>8}")
for m in mats:
    c = int(counts[m["id"]])
    print(f"{m['id']:>2}  {m['name']:<20}{c:>11,}{100*c/M.size:7.1f}%")
d = [round(s * float(man["cell_size_m"]), 1) for s in M.shape]
print(f"\nphysical extent: {d[0]} x {d[1]} x {d[2]} m  (X x Y-up x Z)")

## Tri-plane budget

How this one volume decomposes into the three axis-aligned slice stacks. Each stack's
interior slices become 2-D training planes for one network; stacking them rebuilds the
volume. This is where a single expensive 3-D solve pays for many cheap 2-D samples.

In [ ]:
#@title Tri-plane budget
print(f"{'orient':>6}{'plane (H,W)':>15}{'slices':>8}{'interior':>10}")
per_tx = 0
for o in D.PLANE_ORIENTS:
    H, W = D.plane_shape(M.shape, o)
    n = D.n_slices(M.shape, o)
    interior = sum(int(D.slice_plane(inside, o, k).any()) for k in range(n))
    per_tx += interior
    print(f"{o:>6}{str((H, W)):>15}{n:>8}{interior:>10}")
print(f"\n~{per_tx} interior training planes per Tx across the three stacks")
print("(sliced on the fly by phase_c3_planes_train.ipynb — no extra storage).")

## Sample Tx positions + splits

Split **by position** over X-Z octants (a Tx never appears in two splits). Reuses
`dataset/splits.json` if it already matches this scene, else samples fresh — so a new
voxelized scene bootstraps its own splits.

In [ ]:
#@title Sample / load splits
splits_file = f"{OUT}/splits.json"
_sha = D.scene_sha(M)
if os.path.exists(splits_file):
    sp = json.load(open(splits_file))
    if sp.get("scene_sha") not in (None, _sha):
        raise SystemExit("splits.json is for a different scene — delete it to resample.")
    print(f"reusing splits.json: {len(sp['positions'])} positions "
          f"({len(sp['train'])}/{len(sp['val'])}/{len(sp['test'])})")
else:
    cap = D.max_positions(valid, MIN_SPACING)
    pos = D.sample_tx_positions(valid, min(N_POSITIONS, cap), MIN_SPACING, seed=SEED)
    parts = D.make_splits(pos, M.shape, seed=SEED + 1)
    sp = dict(scene_sha=_sha, seed=SEED, min_spacing=MIN_SPACING, positions=pos.tolist(), **parts)
    print(f"sampled {len(pos)} / {cap} max positions -> "
          f"{len(sp['train'])}/{len(sp['val'])}/{len(sp['test'])} (train/val/test)")

positions = np.asarray(sp["positions"], float)
N_POSITIONS = min(N_POSITIONS, len(positions))
sp["train_bands_mhz"] = TRAIN_BANDS
sp["n_positions_requested"] = int(N_POSITIONS)
sp["run_mode"] = RUN_MODE
json.dump(sp, open(splits_file, "w"), indent=1)
print(f"using first {N_POSITIONS} positions for this run")

## Preflight — does the target saturate the clip ceiling?

If most interior voxels pin to the norm ceiling the dataset is nearly constant. Refuses
to generate above 35% (usually a `SceneV3.use_satobs` regression).

In [ ]:
#@title Preflight
rep = D.clip_report(scene, man, inside, norm, n_probe=(1 if RUN_MODE == "smoke" else 4),
                    bands=TRAIN_BANDS, seed=0)
print(f"clip ceiling {rep['pl_max_db']:.0f} dB | worst clipped {rep['worst_clipped_fraction']*100:.1f}%")
if rep["worst_clipped_fraction"] > 0.35:
    raise SystemExit("PREFLIGHT FAILED — targets mostly saturate the ceiling; check SceneV3.use_satobs / sat_obs.")
print("preflight OK")

## Generate PL-only volume shards (re-runnable)

One geometric pass per Tx solves all bands; we store the normalized PL volume and skip
the eikonal (`tau=None` -> no `_tau.npy`). The shard guard from the completeness fix
regenerates any slot that doesn't match this run (bands / scene / smoke leftovers).

In [ ]:
#@title Solve + write shards
band_idx = [scene.band_index(f) for f in TRAIN_BANDS]
n_shards = int(np.ceil(N_POSITIONS / SHARD_POS))

_ok, _rows = D.audit_shards(OUT, shard_pos=SHARD_POS, n_positions=N_POSITIONS,
                            bands=TRAIN_BANDS, scene_sha=_sha)
_bad = [r for r in _rows if r["status"] != "ok"]
print(f"{len(_bad)}/{len(_rows)} shard slots need (re)generating" if _bad
      else "all shard slots present and matching this run")

t0, done = time.time(), 0
for s in range(n_shards):
    p0, p1 = s * SHARD_POS, min((s + 1) * SHARD_POS, N_POSITIONS)
    if D.shard_complete(OUT, s, expect_pos=p1 - p0, bands=TRAIN_BANDS, scene_sha=_sha):
        print(f"shard {s:03d}: complete, skipping"); continue
    if os.path.exists(D.shard_paths(OUT, s)["meta"]):
        print(f"shard {s:03d}: present but does not match this run - regenerating")
    pl_rows = []; m_tx, m_f, m_ff, m_pos = [], [], [], []
    for pi in range(p0, p1):
        tx = tuple(float(v) for v in positions[pi])
        PL = scene.pathloss_maps(tx)                     # PL only, all bands, no eikonal
        for f, bi in zip(TRAIN_BANDS, band_idx):
            pl_rows.append(norm.pl_to_norm(PL[bi]))
            m_tx.append(positions[pi]); m_f.append(f)
            m_ff.append(norm.freq_feature(f)); m_pos.append(pi)
        done += 1
    D.write_shard(OUT, s, np.stack(pl_rows), None, dict(
        tx=np.array(m_tx, np.int16), freq_mhz=np.array(m_f, np.float32),
        freq_feat=np.array(m_ff, np.float32), pos_id=np.array(m_pos, np.int32),
        scene_sha=_sha, bands_mhz=np.array(TRAIN_BANDS, np.float32)))
    rate = (time.time() - t0) / max(done, 1)
    print(f"shard {s+1:3d}/{n_shards}  pos {p0}-{p1-1}  {rate:.1f} s/pos  "
          f"ETA {(N_POSITIONS-p1)*rate/60:.0f} min")

print(f"\ndone: {len(D.list_shards(OUT))} PL-only shards in {(time.time()-t0)/60:.1f} min")

## Tri-plane sanity — one Tx sliced into xy / yz / zx

Reads a generated sample back, checks the slice/stack round-trip is exact for all three
orientations (the premise of the ensemble), and shows the plane through the Tx in each.

In [ ]:
#@title Sanity view
import matplotlib.pyplot as plt
pl, tau, meta = D.open_shard(OUT, 0)          # tau is None (PL-only)
j = 0
tx = tuple(int(v) for v in meta["tx"][j]); f = float(meta["freq_mhz"][j])
vol = norm.norm_to_pl(np.asarray(pl[j], np.float32))     # (X,Y,Z) dB

for o in D.PLANE_ORIENTS:                     # stacking slices rebuilds the volume exactly
    reb = D.stack_planes([D.slice_plane(vol, o, k) for k in range(D.n_slices(M.shape, o))], o)
    assert np.array_equal(reb, vol), o
print("slice/stack round-trip exact for xy, yz, zx")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, o in zip(ax, D.PLANE_ORIENTS):
    fa = D._ORIENT_FIXED_AXIS[o]; k = int(tx[fa])
    img = np.where(D.slice_plane(inside, o, k), D.slice_plane(vol, o, k), np.nan)
    im = a.imshow(img.T, origin="lower", cmap="viridis")
    a.set_title(f"{o}-plane at {['X', 'Y', 'Z'][fa]}={k}"); fig.colorbar(im, ax=a, label="dB")
plt.suptitle(f"Tx {list(tx)}  @ {f:.0f} MHz"); plt.tight_layout(); plt.show()
print("\nDataset ready (PL-only, tri-plane). Next: phase_c3_planes_train.ipynb")